# BaseAttentive as a Kernel for Robust Neural Networks

This notebook demonstrates using BaseAttentive as a kernel component within larger neural network architectures for building robust, production-ready forecasting systems.

## Table of Contents
1. Ensemble Methods with Multiple BaseAttentive Models
2. Physics-Guided Neural Networks with BaseAttentive
3. Transfer Learning and Domain Adaptation
4. Multi-Task Learning for Correlated Predictions

---

## Setup and Imports

In [ ]:
import warnings

import numpy as np
import tensorflow as tf

warnings.filterwarnings("ignore")

# BaseAttentive kernel
from base_attentive import BaseAttentive
from base_attentive.backend import set_backend

set_backend("tensorflow")

print("Imports successful!")

## 1. Ensemble Methods: Robust Predictions via BaseAttentive Kernels

### Concept
Combine multiple BaseAttentive models with different architectures to reduce overfitting and improve robustness:
- Model 1: Hybrid (attention + LSTM)
- Model 2: Pure transformer (self-attention only)
- Model 3: LSTM-heavy (attention heads tuned to past)

### Architecture
```
Inputs (static, dynamic_past, future)
    ↓
BaseAttentive Kernel 1 → predictions_1
    ↓ (parallel)
BaseAttentive Kernel 2 → predictions_2
    ↓ (parallel)
BaseAttentive Kernel 3 → predictions_3
    ↓
Ensemble Layer (weighted average or meta-learner)
    ↓
Final Predictions (robust)
```

In [ ]:
# Create sample data
np.random.seed(42)
batch_size = 32
static_features = np.random.randn(batch_size, 4)
dynamic_past = np.random.randn(batch_size, 168, 5)
known_future = np.random.randn(batch_size, 24, 2)
target = np.random.randn(batch_size, 24, 1)

# Create individual BaseAttentive kernels
kernel_1 = BaseAttentive(
    static_dim=4,
    dynamic_dim=5,
    future_dim=2,
    forecast_horizon=24,
    lookback_window=168,
    mode="hybrid",
    attention_stack=[
        ("cross_attention", {"heads": 4, "dim": 64}),
        ("self_attention", {"heads": 4, "dim": 64}),
    ],
    name="kernel_hybrid",
)

kernel_2 = BaseAttentive(
    static_dim=4,
    dynamic_dim=5,
    future_dim=2,
    forecast_horizon=24,
    lookback_window=168,
    mode="transformer",
    attention_stack=[
        ("self_attention", {"heads": 8, "dim": 128}),
        ("self_attention", {"heads": 8, "dim": 128}),
    ],
    name="kernel_transformer",
)

kernel_3 = BaseAttentive(
    static_dim=4,
    dynamic_dim=5,
    future_dim=2,
    forecast_horizon=24,
    lookback_window=168,
    mode="hybrid",
    attention_stack=[
        ("memory_attention", {"heads": 4, "dim": 64, "memory_size": 32}),
        ("cross_attention", {"heads": 4, "dim": 64}),
    ],
    name="kernel_memory",
)

print("Created 3 BaseAttentive kernels for ensemble")

In [ ]:
# Build ensemble model using Keras functional API
# Inputs
static_input = tf.keras.Input(shape=(4,), name="static")
dynamic_input = tf.keras.Input(shape=(168, 5), name="dynamic_past")
future_input = tf.keras.Input(shape=(24, 2), name="known_future")

# Kernel predictions (parallel)
pred_1 = kernel_1([static_input, dynamic_input, future_input])
pred_2 = kernel_2([static_input, dynamic_input, future_input])
pred_3 = kernel_3([static_input, dynamic_input, future_input])

# Ensemble combination - learnable weights
ensemble_concat = tf.keras.layers.Concatenate(axis=-1)([pred_1, pred_2, pred_3])
ensemble_combined = tf.keras.layers.Dense(64, activation="relu")(ensemble_concat)
ensemble_output = tf.keras.layers.Dense(24, activation="linear")(ensemble_combined)
ensemble_output = tf.keras.layers.Reshape((24, 1))(ensemble_output)

# Create ensemble model
ensemble_model = tf.keras.Model(
    inputs=[static_input, dynamic_input, future_input],
    outputs=ensemble_output,
    name="BaseAttentive_Ensemble",
)

ensemble_model.compile(optimizer="adam", loss="mse", metrics=["mae"])

print("\nEnsemble Architecture:")
ensemble_model.summary()

In [ ]:
# Train ensemble
print("Training ensemble model...")
history_ensemble = ensemble_model.fit(
    [static_features, dynamic_past, known_future],
    target,
    epochs=3,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
)

# Make predictions
ensemble_predictions = ensemble_model.predict(
    [static_features[:5], dynamic_past[:5], known_future[:5]]
)

print("\nEnsemble Results:")
print(f"Prediction shape: {ensemble_predictions.shape}")
print(f"Mean prediction: {np.mean(ensemble_predictions):.4f}")
print(f"Prediction variance: {np.var(ensemble_predictions):.4f}")

## 2. Physics-Guided Networks: BaseAttentive + Physics Constraints

### Concept
Incorporate domain knowledge and physical constraints into the neural network:
- BaseAttentive learns data-driven patterns
- Physics layer enforces conservation laws
- Hybrid loss combines data loss + physics constraints

### Example: Energy Conservation Constraint
```
Energy_out = Energy_in + Solar_Production - Losses
Physics Loss = MSE on this constraint
Total Loss = Data Loss + λ × Physics Loss
```

In [ ]:
def physics_constraint_energy(predictions, static_features, dynamic_past, known_future):
    """
    Apply energy conservation constraint:
    Energy_demand_t+1 ≈ Energy_demand_t × decay_factor + Solar_available
    """
    # Extract from inputs
    decay_factor = 0.95  # Natural decay

    # Get previous energy level
    prev_energy = dynamic_past[:, -1:, 0:1]  # Last value from dynamic past

    # Solar available from known future (normalized)
    solar_available = known_future[:, :, 0:1]

    # Physics constraint prediction
    physics_pred = prev_energy * decay_factor + solar_available * 100

    return tf.reduce_mean(tf.square(predictions - physics_pred))


# Physics-guided model using custom training loop
class PhysicsGuidedEnsemble(tf.keras.Model):
    def __init__(self, ensemble_model, physics_weight=0.1):
        super(PhysicsGuidedEnsemble, self).__init__()
        self.ensemble = ensemble_model
        self.physics_weight = physics_weight

    def call(self, inputs):
        static, dynamic, future = inputs
        return self.ensemble([static, dynamic, future])

    def train_step(self, data):
        x, y = data
        static, dynamic, future = x

        with tf.GradientTape() as tape:
            # Data loss
            predictions = self.ensemble([static, dynamic, future], training=True)
            data_loss = tf.keras.losses.mse(y, predictions)
            data_loss = tf.reduce_mean(data_loss)

            # Physics constraint loss
            physics_loss = physics_constraint_energy(
                predictions, static, dynamic, future
            )

            # Combined loss
            total_loss = data_loss + self.physics_weight * physics_loss

        # Update weights
        trainable_vars = self.trainable_variables
        gradients = tape.gradient(total_loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        return {
            "loss": total_loss,
            "data_loss": data_loss,
            "physics_loss": physics_loss,
        }


# Create physics-guided model
pg_model = PhysicsGuidedEnsemble(ensemble_model, physics_weight=0.05)
pg_model.compile(optimizer="adam")

print("Physics-guided model created!")

In [ ]:
# Train physics-guided model
print("Training physics-guided model...")
history_pg = pg_model.fit(
    ([static_features, dynamic_past, known_future], target),
    epochs=3,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
)

# Make predictions
pg_predictions = pg_model.predict(
    [static_features[:5], dynamic_past[:5], known_future[:5]]
)

print("\nPhysics-Guided Model Results:")
print(f"Prediction shape: {pg_predictions.shape}")
print("Predictions respect physics constraints: ✓")

## 3. Transfer Learning: Pre-Train on Large Dataset, Fine-Tune on Task

### Concept
- Pre-train BaseAttentive on large multi-site dataset
- Fine-tune on specific location/application
- Beneficial when task-specific data is limited

### Strategy
1. Pre-train on 100+ locations for general patterns
2. Freeze encoder layers
3. Fine-tune decoder on target location
4. Gradually unfreeze deeper layers

In [ ]:
# Create pre-trained BaseAttentive model
pretrained_model = BaseAttentive(
    static_dim=4,
    dynamic_dim=5,
    future_dim=2,
    forecast_horizon=24,
    lookback_window=168,
    mode="hybrid",
    attention_stack=[
        ("cross_attention", {"heads": 8, "dim": 128}),
        ("self_attention", {"heads": 8, "dim": 128}),
    ],
    name="pretrained_encoder",
)

pretrained_model.compile(optimizer="adam", loss="mse")

# Pre-train on large dataset (simulated)
large_dataset_size = 128
static_large = np.random.randn(large_dataset_size, 4)
dynamic_large = np.random.randn(large_dataset_size, 168, 5)
future_large = np.random.randn(large_dataset_size, 24, 2)
target_large = np.random.randn(large_dataset_size, 24, 1)

print("Pre-training on large dataset...")
history_pretrain = pretrained_model.fit(
    [static_large, dynamic_large, future_large],
    target_large,
    epochs=2,
    batch_size=32,
    verbose=0,
)

print("Pre-training complete!")

In [ ]:
# Create transfer learning model with frozen encoder
transfer_model = tf.keras.models.clone_model(pretrained_model)

# Freeze most layers
for layer in transfer_model.layers[:-4]:  # Freeze all but last 4 layers
    layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss="mse"
)

# Fine-tune on target location (small dataset)
target_static = np.random.randn(16, 4)
target_dynamic = np.random.randn(16, 168, 5)
target_future = np.random.randn(16, 24, 2)
target_y = np.random.randn(16, 24, 1)

print("Fine-tuning on target location...")
history_finetune = transfer_model.fit(
    [target_static, target_dynamic, target_future],
    target_y,
    epochs=5,
    batch_size=8,
    verbose=0,
)

print("Fine-tuning complete!")

# Make predictions
transfer_pred = transfer_model.predict(
    [target_static[:3], target_dynamic[:3], target_future[:3]]
)
print(f"Transfer learning predictions shape: {transfer_pred.shape}")

## 4. Multi-Task Learning: Predict Correlated Quantities Jointly

### Concept
Train single BaseAttentive model to predict multiple correlated targets:
- Task 1: Energy demand
- Task 2: Peak hour prediction
- Task 3: Anomaly detection

### Benefits
- Shared representations reduce overfitting
- Different tasks provide regularization
- Single inference for all predictions

In [ ]:
# Create multi-task BaseAttentive model
static_in = tf.keras.Input(shape=(4,))
dynamic_in = tf.keras.Input(shape=(168, 5))
future_in = tf.keras.Input(shape=(24, 2))

# Shared BaseAttentive backbone
base_model = BaseAttentive(
    static_dim=4,
    dynamic_dim=5,
    future_dim=2,
    forecast_horizon=24,
    lookback_window=168,
    mode="hybrid",
    attention_stack=[
        ("cross_attention", {"heads": 8, "dim": 128}),
        ("self_attention", {"heads": 8, "dim": 128}),
    ],
)

shared_output = base_model([static_in, dynamic_in, future_in])

# Task 1: Energy demand prediction
task1_dense = tf.keras.layers.Dense(128, activation="relu")(shared_output)
task1_output = tf.keras.layers.Dense(24, activation="linear", name="energy_demand")(
    task1_dense
)

# Task 2: Peak hour prediction (classification)
task2_dense = tf.keras.layers.Dense(64, activation="relu")(shared_output)
task2_output = tf.keras.layers.Dense(24, activation="softmax", name="peak_hour")(
    task2_dense
)

# Task 3: Anomaly detection score
task3_dense = tf.keras.layers.Dense(64, activation="relu")(shared_output)
task3_output = tf.keras.layers.Dense(24, activation="sigmoid", name="anomaly_score")(
    task3_dense
)

# Multi-task model
multitask_model = tf.keras.Model(
    inputs=[static_in, dynamic_in, future_in],
    outputs=[task1_output, task2_output, task3_output],
    name="BaseAttentive_MultiTask",
)

# Compile with task-specific losses
multitask_model.compile(
    optimizer="adam",
    loss={
        "energy_demand": "mse",
        "peak_hour": "categorical_crossentropy",
        "anomaly_score": "binary_crossentropy",
    },
    loss_weights={"energy_demand": 2.0, "peak_hour": 1.0, "anomaly_score": 0.5},
    metrics=["mae"],
)

print("Multi-task model created!")
multitask_model.summary()

In [ ]:
# Create target data for all 3 tasks
task1_target = np.random.randn(batch_size, 24)  # Energy demand
task2_target = tf.keras.utils.to_categorical(
    np.random.randint(0, 24, batch_size), num_classes=24
)  # Peak hour (dummy)
task3_target = np.random.rand(batch_size, 24)  # Anomaly scores

# Train multi-task model
print("Training multi-task model...")
history_multitask = multitask_model.fit(
    [static_features, dynamic_past, known_future],
    [task1_target, task2_target, task3_target],
    epochs=3,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
)

print("Training complete!")

# Make predictions
mt_pred1, mt_pred2, mt_pred3 = multitask_model.predict(
    [static_features[:3], dynamic_past[:3], known_future[:3]]
)

print("\nMulti-Task Predictions:")
print(f"  Energy demand shape: {mt_pred1.shape}")
print(f"  Peak hour shape: {mt_pred2.shape}")
print(f"  Anomaly score shape: {mt_pred3.shape}")

## Summary: Kernel Architectures

BaseAttentive works effectively as a neural network kernel:

| Architecture | Purpose | Advantages |
|---|---|---|
| **Ensemble** | Robust predictions | Reduced variance, better generalization |
| **Physics-Guided** | Respect domain constraints | Physically plausible predictions |
| **Transfer Learning** | Few-shot adaptation | Leverage large pre-trained models |
| **Multi-Task** | Correlated predictions | Shared representations, regularization |

### Key Insights
- ✓ BaseAttentive as core component maintains flexibility
- ✓ Combine with standard Keras layers for custom architectures
- ✓ Each kernel can be independently trained or frozen
- ✓ Supports custom training loops for advanced objectives
- ✓ GPU acceleration through Keras backend